### 1. Import Library

In [ ]:
import gdown
import zipfile
import os
import shutil
import random

### 2. Persiapan Dataset

Download dataset dari Google Drive

In [ ]:
file_id = '1o0b6Nqs89zYoyRcnih5oGS7sk0bIrImo'
url = f'https://drive.google.com/uc?id={file_id}'
output_zip = 'dataset.zip'

print("Mengunduh dataset dari Google Drive...")
gdown.download(url, output_zip, quiet=False)

Extract Dataset

In [ ]:
extract_dir = 'dataset'
print("\nMengekstrak dataset...")
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)


### 3. Membuat File `NCCTDataset`

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from PIL import Image
import random

class NCCTDataset(Dataset):
    def __init__(self, split_dir, limit_size=None, transform=None):
        self.split_dir = split_dir
        self.transform = transform

        img_dir = os.path.join(split_dir, 'images')
        mask_dir = os.path.join(split_dir, 'masks')

        self.image_paths = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir)])
        self.mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])

        if limit_size is not None:
            self.image_paths = self.image_paths[:limit_size]
            self.mask_paths = self.mask_paths[:limit_size]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            if self.split_dir.endswith('train'):
                if random.random() > 0.5:
                    img = TF.hflip(img)
                    mask = TF.hflip(mask)
                if random.random() > 0.5:
                    img = TF.vflip(img)
                    mask = TF.vflip(mask)
                angle = random.uniform(-15, 15)
                img = TF.rotate(img, angle)
                mask = TF.rotate(mask, angle)

            img = self.transform(img)
            mask = self.transform(mask)

        # FIX: 'mask' is already a tensor here, don't call ToTensor() again
        label = 1.0 if torch.max(mask) > 0 else 0.0

        return img, mask, torch.tensor([label], dtype=torch.float32)

Cek Dataset

In [ ]:
import matplotlib.pyplot as plt

def get_dataloaders(base_dir, batch_size=4, limit_size=None):
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ])

    train_dataset = NCCTDataset(os.path.join(base_dir, 'train'), limit_size=limit_size, transform=transform)
    val_dataset = NCCTDataset(os.path.join(base_dir, 'val'), limit_size=limit_size, transform=transform)
    test_dataset = NCCTDataset(os.path.join(base_dir, 'test'), limit_size=limit_size, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# Ambil 1 batch data untuk verifikasi
train_loader, _, _ = get_dataloaders('dataset', batch_size=4)
images, masks, labels = next(iter(train_loader))

print(f"Image shape: {images.shape}")
print(f"Mask shape: {masks.shape}")
print(f"Max mask value: {masks.max():.2f}")
print(f"Min mask value: {masks.min():.2f}")

# Visualisasi sederhana
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(images[0][0], cmap='gray'); plt.title("Input Image")
plt.subplot(1, 2, 2); plt.imshow(masks[0][0], cmap='gray'); plt.title("Ground Truth Mask")
plt.show()

### 4. Membuat Model (CSPNet Architecture)

CSPBlock

In [ ]:
class CSPBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid_channels = out_channels // 2

        self.main_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.skip_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True)
        )

        self.transition = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        main_out = self.main_conv(x)
        skip_out = self.skip_conv(x)
        merged = torch.cat([main_out, skip_out], dim=1)
        return self.transition(merged)

Main Model

In [ ]:
class NCCTSegmentationModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder (Backbone)
        self.initial_conv = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.layer1 = CSPBlock(32, 64)   # Output: 64 channels
        self.layer2 = CSPBlock(64, 128)  # Output: 128 channels
        self.layer3 = CSPBlock(128, 256) # Output: 256 channels

        self.pool = nn.MaxPool2d(2, 2)

        # Decoder with Skip Connections
        # features (256 channels) -> upconv1 (128 channels)
        self.upconv1 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        # concat(up1:128, layer3:256) = 384
        self.decoder1 = CSPBlock(128 + 256, 128)

        self.upconv2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        # concat(up2:64, layer2:128) = 192
        self.decoder2 = CSPBlock(64 + 128, 64)

        self.upconv3 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        # concat(up3:32, layer1:64) = 96
        self.decoder3 = CSPBlock(32 + 64, 32)

        self.seg_output = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        x0 = F.relu(self.initial_conv(x))

        x1 = self.layer1(x0)
        p1 = self.pool(x1)

        x2 = self.layer2(p1)
        p2 = self.pool(x2)

        x3 = self.layer3(p2)
        features = self.pool(x3)

        # Decoder
        d1 = self.upconv1(features)
        d1 = torch.cat([d1, x3], dim=1)
        d1 = self.decoder1(d1)

        d2 = self.upconv2(d1)
        d2 = torch.cat([d2, x2], dim=1)
        d2 = self.decoder2(d2)

        d3 = self.upconv3(d2)
        d3 = torch.cat([d3, x1], dim=1)
        d3 = self.decoder3(d3)

        seg_out = self.seg_output(d3)
        return seg_out

### 4b. Additional Models (UNet, Attention UNet, ResUNet)

In [ ]:
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = UNetBlock(in_channels, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.enc4 = UNetBlock(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = UNetBlock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = UNetBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        d4 = self.up4(e4)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        
        return self.out(d2)

In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = UNetBlock(in_channels, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.enc4 = UNetBlock(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.att4 = AttentionGate(F_g=256, F_l=256, F_int=128)
        self.dec4 = UNetBlock(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att3 = AttentionGate(F_g=128, F_l=128, F_int=64)
        self.dec3 = UNetBlock(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att2 = AttentionGate(F_g=64, F_l=64, F_int=32)
        self.dec2 = UNetBlock(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        d4 = self.up4(e4)
        x3 = self.att4(g=d4, x=e3)
        d4 = torch.cat([d4, x3], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.up3(d4)
        x2 = self.att3(g=d3, x=e2)
        d3 = torch.cat([d3, x2], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        x1 = self.att2(g=d2, x=e1)
        d2 = torch.cat([d2, x1], dim=1)
        d2 = self.dec2(d2)
        
        return self.out(d2)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv(x)
        out += residual
        return self.relu(out)

class ResUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = ResidualBlock(in_channels, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ResidualBlock(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        
        d3 = self.up3(e3)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        
        return self.out(d2)

### 5. Mendefinisikan Fungsi Loss

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super(BCEDiceLoss, self).__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]).cuda())

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        inputs_sigmoid = torch.sigmoid(inputs)
        intersection = (inputs_sigmoid * targets).sum(dim=(2, 3))
        union = inputs_sigmoid.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice_score = (2. * intersection + 1e-6) / (union + 1e-6)
        dice_loss = 1.0 - dice_score.mean()
        return (self.bce_weight * bce_loss) + ((1 - self.bce_weight) * dice_loss)

def dice_coeff(pred, target, threshold=0.5, epsilon=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()
    inter = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2. * inter + epsilon) / (union + epsilon)
    return dice.mean().item()

### 6. Melatih Model

Fungsi `train_segmentation_model`

In [ ]:
def train_model(model, train_loader, val_loader, epochs=40, model_name="Model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    criterion = BCEDiceLoss(bce_weight=0.5).cuda()

    print(f"\nTraining {model_name}...")
    print(f"{'Epoch':<8} | {'T-Loss':<8} | {'V-Loss':<8} | {'V-Dice':<8} | {'LR':<8}")
    print("-" * 55)

    best_dice = 0
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, masks, _ in train_loader:
            images, masks = images.cuda(), masks.cuda()
            optimizer.zero_grad()
            seg_out = model(images)
            loss = criterion(seg_out, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss, val_dice = 0, 0
        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.cuda(), masks.cuda()
                seg_out = model(images)
                loss = criterion(seg_out, masks)
                val_loss += loss.item()
                val_dice += dice_coeff(seg_out, masks)

        avg_t_loss = train_loss / len(train_loader)
        avg_v_loss = val_loss / len(val_loader)
        avg_v_dice = val_dice / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        scheduler.step(avg_v_loss)

        print(f"{epoch+1:<8} | {avg_t_loss:<8.4f} | {avg_v_loss:<8.4f} | {avg_v_dice:<8.4f} | {current_lr:<8.6f}")
        
        if avg_v_dice > best_dice:
            best_dice = avg_v_dice
            torch.save(model.state_dict(), f"{model_name.lower().replace(' ', '_')}_best.pth")

    return model

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders('dataset')

models_to_train = {
    "CSPNet": NCCTSegmentationModel(),
    "UNet": UNet(),
    "AttentionUNet": AttentionUNet(),
    "ResUNet": ResUNet()
}

trained_models = {}

for name, model in models_to_train.items():
    model = model.cuda()
    trained_models[name] = train_model(model, train_loader, val_loader, epochs=30, model_name=name)

### 8. Evaluasi Model

In [ ]:
def evaluate_all_models(models_dict, test_loader):
    results = {}
    print(f"\n{'Model':<15} | {'Test Dice Score':<15}")
    print("-" * 35)
    
    for name, model in models_dict.items():
        model.eval()
        test_dice = 0
        with torch.no_grad():
            for images, masks, _ in test_loader:
                images, masks = images.cuda(), masks.cuda()
                seg_out = model(images)
                test_dice += dice_coeff(seg_out, masks)
        
        final_dice = test_dice / len(test_loader)
        results[name] = final_dice
        print(f"{name:<15} | {final_dice:<15.4f}")
    
    return results

In [ ]:
evaluation_results = evaluate_all_models(trained_models, test_loader)

### 9. Visualisasi Prediksi vs Ground Truth
Bagian ini membantu untuk melihat secara visual performa model pada data testing.

In [ ]:
def visualize_all_models(models_dict, test_loader, num_samples=3):
    model_names = list(models_dict.keys())
    num_models = len(model_names)
    
    plt.figure(figsize=(20, num_samples * 4))
    
    with torch.no_grad():
        for i in range(num_samples):
            images, masks, _ = next(iter(test_loader))
            images_gpu = images.cuda()
            
            # Plot Input
            plt.subplot(num_samples, num_models + 2, i * (num_models + 2) + 1)
            plt.imshow(images[0][0].cpu().numpy(), cmap='gray')
            plt.title("Input")
            plt.axis('off')
            
            # Plot Ground Truth
            plt.subplot(num_samples, num_models + 2, i * (num_models + 2) + 2)
            plt.imshow(masks[0][0].cpu().numpy(), cmap='gray')
            plt.title("GT")
            plt.axis('off')
            
            # Plot each model's prediction
            for j, name in enumerate(model_names):
                model = models_dict[name]
                model.eval()
                out = model(images_gpu)
                pred = (torch.sigmoid(out) > 0.5).cpu().numpy()[0][0]
                
                plt.subplot(num_samples, num_models + 2, i * (num_models + 2) + 3 + j)
                plt.imshow(pred, cmap='gray')
                plt.title(name)
                plt.axis('off')
                
    plt.tight_layout()
    plt.show()

### 4b. Additional Models (UNet, Attention UNet, ResUNet)

In [ ]:
class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = UNetBlock(in_channels, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.enc4 = UNetBlock(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = UNetBlock(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = UNetBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = UNetBlock(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        d4 = self.up4(e4)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.up3(d4)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        
        return self.out(d2)

In [ ]:
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = UNetBlock(in_channels, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.enc4 = UNetBlock(256, 512)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.att4 = AttentionGate(F_g=256, F_l=256, F_int=128)
        self.dec4 = UNetBlock(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.att3 = AttentionGate(F_g=128, F_l=128, F_int=64)
        self.dec3 = UNetBlock(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.att2 = AttentionGate(F_g=64, F_l=64, F_int=32)
        self.dec2 = UNetBlock(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        d4 = self.up4(e4)
        x3 = self.att4(g=d4, x=e3)
        d4 = torch.cat([d4, x3], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.up3(d4)
        x2 = self.att3(g=d3, x=e2)
        d3 = torch.cat([d3, x2], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        x1 = self.att2(g=d2, x=e1)
        d2 = torch.cat([d2, x1], dim=1)
        d2 = self.dec2(d2)
        
        return self.out(d2)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv(x)
        out += residual
        return self.relu(out)

class ResUNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        self.enc1 = ResidualBlock(in_channels, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        
        self.pool = nn.MaxPool2d(2)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ResidualBlock(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ResidualBlock(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        
        d3 = self.up3(e3)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.up2(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        
        return self.out(d2)